In [1]:
!nvidia-smi

Mon May 18 13:46:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


# 1. 设置超参数
batch_size = 128
learning_rate = 0.001
epochs = 10


# 2. 设置设备
device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"

print("Using device:", device)
print("Using AMP:", use_amp)


# 3. 数据预处理
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])


# 4. 下载 MNIST 数据集
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)


# 5. 创建 DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True if device == "cuda" else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True if device == "cuda" else False
)


# 6. 定义 3 层 MLP
class MLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.net(x)


model = MLP().to(device)


# 7. 定义损失函数、优化器和 AMP Scaler
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


# 8. 训练函数
def train_one_epoch(epoch):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    print(
        f"Epoch [{epoch}/{epochs}] "
        f"Train Loss: {avg_loss:.4f}, "
        f"Train Acc: {accuracy:.4f}"
    )


# 9. 测试函数
def evaluate(epoch):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                outputs = model(images)
                loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    print(
        f"Epoch [{epoch}/{epochs}] "
        f"Test  Loss: {avg_loss:.4f}, "
        f"Test  Acc: {accuracy:.4f}"
    )


# 10. 主训练循环
for epoch in range(1, epochs + 1):
    train_one_epoch(epoch)
    evaluate(epoch)


Using device: cpu
Using AMP: False


100%|██████████| 9.91M/9.91M [00:02<00:00, 4.62MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 154kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.49MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.55MB/s]
/var/folders/xf/nhn270z56f7f9mxr1zfvbphr0000gn/T/ipykernel_63024/2470869697.py:87: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/var/folders/xf/nhn270z56f7f9mxr1zfvbphr0000gn/T/ipykernel_63024/2470869697.py:104: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch [1/10] Train Loss: 0.2649, Train Acc: 0.9223


/var/folders/xf/nhn270z56f7f9mxr1zfvbphr0000gn/T/ipykernel_63024/2470869697.py:141: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch [1/10] Test  Loss: 0.1209, Test  Acc: 0.9618
Epoch [2/10] Train Loss: 0.1023, Train Acc: 0.9684
Epoch [2/10] Test  Loss: 0.0993, Test  Acc: 0.9667
Epoch [3/10] Train Loss: 0.0684, Train Acc: 0.9785
Epoch [3/10] Test  Loss: 0.0766, Test  Acc: 0.9762
Epoch [4/10] Train Loss: 0.0505, Train Acc: 0.9842
Epoch [4/10] Test  Loss: 0.0761, Test  Acc: 0.9774
Epoch [5/10] Train Loss: 0.0397, Train Acc: 0.9875
Epoch [5/10] Test  Loss: 0.0790, Test  Acc: 0.9771
Epoch [6/10] Train Loss: 0.0303, Train Acc: 0.9905
Epoch [6/10] Test  Loss: 0.0772, Test  Acc: 0.9773
Epoch [7/10] Train Loss: 0.0265, Train Acc: 0.9915
Epoch [7/10] Test  Loss: 0.0780, Test  Acc: 0.9784
Epoch [8/10] Train Loss: 0.0231, Train Acc: 0.9921
Epoch [8/10] Test  Loss: 0.0744, Test  Acc: 0.9787
Epoch [9/10] Train Loss: 0.0195, Train Acc: 0.9932
Epoch [9/10] Test  Loss: 0.0816, Test  Acc: 0.9772
Epoch [10/10] Train Loss: 0.0170, Train Acc: 0.9943
Epoch [10/10] Test  Loss: 0.0787, Test  Acc: 0.9795
